# Producer: Exchange API -> S3 Raw Lake (JSONL)
# - Fetch granularity: 1 minute
# - Write frequency: 15-minute file per symbol
# - Supports mode: realtime / backfill


# Cell 1 — Imports

In [0]:
import gzip
import io
import json
import random
import re
import time
import uuid
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Tuple

import boto3
import requests

In [0]:
%skip
dbutils.widgets.removeAll()

# Cell 2 — Widgets

In [0]:
# ==========================================
# 1) Widgets (Parameters)
# ==========================================

# ---- Core ----
dbutils.widgets.text("source", "coinbase")  # coinbase / binance(部分地区不可用)
dbutils.widgets.text("symbols", "BTC-USD,ETH-USD")  # coinbase 常用 BTC-USD
dbutils.widgets.text("interval", "1m")  # 固定 1m（本 producer 只做 1m）
dbutils.widgets.text("mode", "realtime")  # realtime / backfill

# ---- Backfill window (UTC ISO8601) ----
dbutils.widgets.text("start_ts", "2025-01-01T00:00:00Z")  # 例如 2026-02-10T00:00:00Z
dbutils.widgets.text(
    "end_ts", "2025-12-31T00:00:00Z"
)  # 例如 2026-02-11T00:00:00Z（可空：表示到 now）

# ---- Throttle & safety (Producer-grade) ----
dbutils.widgets.text("sleep_ms", "250")  # 控制 API 节奏（rate limit 保护）
dbutils.widgets.text("max_days", "365")  # 回补窗口上限（防止一次回补太大）
dbutils.widgets.text("safety_lag_minutes", "3")  # realtime：避免拿到未稳定/被修正的最新 bar

# ---- S3 target ----
dbutils.widgets.text("s3_bucket", "enterprise-raw-lakehouse")
dbutils.widgets.text("s3_prefix", "crypto/ohlc_1m")  # crypto/ohlc_1m
dbutils.widgets.text("merge_minutes", "60")  # 5/10/15/20/30/60（需整除 60）

# ---- Optional: run control (Job-friendly) ----
dbutils.widgets.text("loop_sleep_seconds", "30")  # realtime：每轮 sleep（避免紧循环）
dbutils.widgets.text("max_loops", "0")  # 0=无限循环；>0 跑 N 次后退出（防僵尸 job）

# ---- AWS Access Key 输入框 ----
dbutils.widgets.text("aws_access_key", "", "AWS Access Key")
dbutils.widgets.text("aws_secret_key", "", "AWS Secret Key")
dbutils.widgets.text("aws_region", "eu-central-1", "AWS Region")

print("✅ 输入框创建完成！请查看页面顶部。")
# ==========================================
# 2) Helpers
# ==========================================

ISO_Z_RE = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")


def _parse_int(name: str, default: int) -> int:
    raw = dbutils.widgets.get(name)
    raw = raw.strip() if raw is not None else ""
    if raw == "":
        return default
    try:
        return int(raw)
    except Exception as e:
        raise ValueError(f"❌ {name} 必须是整数，当前='{raw}'") from e


def _parse_iso_utc_z(name: str) -> datetime:
    """
    Parse 'YYYY-MM-DDTHH:MM:SSZ' into timezone-aware UTC datetime.
    """
    s = dbutils.widgets.get(name).strip()
    if not s:
        raise ValueError(f"❌ {name} 不能为空（需要 UTC ISO8601，例如 2026-02-10T00:00:00Z）")
    if not ISO_Z_RE.match(s):
        raise ValueError(
            f"❌ {name} 格式错误：'{s}'，需要 'YYYY-MM-DDTHH:MM:SSZ'（以 Z 结尾，UTC）"
        )
    # datetime.fromisoformat 不直接吃 'Z'（部分 python 版本），这里手动处理
    dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
    return dt.astimezone(timezone.utc)


def _now_utc() -> datetime:
    return datetime.now(timezone.utc)


def _hours_divisible(minutes: int) -> bool:
    return minutes > 0 and (60 % minutes == 0)


# ==========================================
# 3) Parsing (Get & Normalize)
# ==========================================

source = dbutils.widgets.get("source").strip().lower()
symbols = [s.strip() for s in dbutils.widgets.get("symbols").split(",") if s.strip()]
interval = dbutils.widgets.get("interval").strip().lower()
mode = dbutils.widgets.get("mode").strip().lower()

start_ts_raw = dbutils.widgets.get("start_ts").strip()
end_ts_raw = dbutils.widgets.get("end_ts").strip()

sleep_ms = _parse_int("sleep_ms", 250)
max_days = _parse_int("max_days", 7)
safety_lag_minutes = _parse_int("safety_lag_minutes", 3)

s3_bucket = dbutils.widgets.get("s3_bucket").strip()
s3_prefix = dbutils.widgets.get("s3_prefix").strip().strip("/")
merge_minutes = _parse_int("merge_minutes", 15)

loop_sleep_seconds = _parse_int("loop_sleep_seconds", 30)
max_loops = _parse_int("max_loops", 0)


# ==========================================
# 4) Validation (Safety First)
# ==========================================

# --- core sanity ---
if not source:
    raise ValueError("❌ source 不能为空（例如 coinbase）")

if not symbols:
    raise ValueError("❌ symbols 不能为空（例如 BTC-USD,ETH-USD）")

assert interval == "1m", "❌ interval 仅支持 1m（本 producer 为 1m 专用）"
assert mode in ("realtime", "backfill"), "❌ mode 必须是 realtime 或 backfill"

# --- s3 target ---
if not s3_bucket:
    raise ValueError("❌ s3_bucket 不能为空")
if not s3_prefix:
    raise ValueError("❌ s3_prefix 不能为空（例如 crypto/ohlc_1m）")

# --- merge policy ---
if not _hours_divisible(merge_minutes):
    raise ValueError("❌ merge_minutes 必须能整除 60（推荐：5/10/15/20/30/60）")

# --- throttle & safety ---
if sleep_ms < 0 or sleep_ms > 10_000:
    raise ValueError("❌ sleep_ms 不合理（建议 0~10000）")

if max_days <= 0 or max_days > 365:
    raise ValueError("❌ max_days 不合理（建议 1~365）")

if safety_lag_minutes < 0 or safety_lag_minutes > 60:
    raise ValueError("❌ safety_lag_minutes 不合理（建议 0~60）")

# --- job control ---
if loop_sleep_seconds < 0 or loop_sleep_seconds > 3600:
    raise ValueError("❌ loop_sleep_seconds 不合理（建议 0~3600）")

if max_loops < 0:
    raise ValueError("❌ max_loops 不能为负数（0=无限循环）")

# --- mode-specific checks ---
start_dt = None
end_dt = None

if mode == "backfill":
    # start_ts 必填；end_ts 可选（默认 now）
    start_dt = _parse_iso_utc_z("start_ts")
    end_dt = _parse_iso_utc_z("end_ts") if end_ts_raw else _now_utc()

    if end_dt <= start_dt:
        raise ValueError(
            f"❌ end_ts 必须大于 start_ts：{start_dt.isoformat()} -> {end_dt.isoformat()}"
        )

    # 回补窗口保护：不允许一次回补超过 max_days
    if end_dt - start_dt > timedelta(days=max_days):
        raise ValueError(
            f"❌ 回补窗口过大：{(end_dt - start_dt)}，超过 max_days={max_days} 天。"
            "请缩小 start_ts/end_ts，或明确提高 max_days（不建议过大）。"
        )

elif mode == "realtime":
    # realtime 不强制 start/end；但 safety_lag 提醒
    pass


# ==========================================
# 5) Summary (Execution Confirmation)
# ==========================================

print("=" * 52)
print(f"🚀 PRODUCER CONFIG  |  MODE: {mode.upper()}")
print("-" * 52)
print(f"🔹 Source:          {source}")
print(f"🔹 Symbols:         {symbols}")
print(f"🔹 Interval:        {interval}")
print(f"🔹 Target S3:       s3://{s3_bucket}/{s3_prefix}/")
print(f"🔹 Merge policy:    every {merge_minutes} minutes (files merged per window)")
print("-" * 52)

if mode == "realtime":
    limit_msg = "Infinite" if max_loops == 0 else f"{max_loops} loops"
    print("🟢 Realtime controls")
    print(f"   • loop_sleep_seconds: {loop_sleep_seconds}s")
    print(f"   • max_loops:          {limit_msg}")
    print(f"   • safety_lag_minutes: {safety_lag_minutes} (avoid unstable latest bars)")
    print(f"   • sleep_ms:           {sleep_ms} (API throttle)")
else:
    print("🔵 Backfill window (UTC)")
    print(f"   • start_ts: {start_dt.isoformat().replace('+00:00', 'Z')}")
    print(f"   • end_ts:   {end_dt.isoformat().replace('+00:00', 'Z')}")
    print(f"   • max_days: {max_days} (guardrail)")
    print(f"   • sleep_ms: {sleep_ms} (API throttle)")

print("=" * 52)

# Cell 3 — Time utilities (windowing)

In [0]:
ISO_Z_RE = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")


def parse_utc_iso(s: str) -> datetime:
    """
    Parse ISO8601 time string into timezone-aware UTC datetime.

    Preferred: 'YYYY-MM-DDTHH:MM:SSZ' (UTC)
    Accepts:   with explicit offset like '+00:00'
    If no tzinfo: assume UTC (but better to pass Z in backfill)
    """
    if not s or not s.strip():
        raise ValueError("timestamp string is empty")

    s = s.strip()

    # Optional strictness: encourage Z format
    # If you want strict only, uncomment next lines:
    # if not (s.endswith("Z") and ISO_Z_RE.match(s)):
    #     raise ValueError(f"Invalid UTC Z format: {s}, expected YYYY-MM-DDTHH:MM:SSZ")

    if s.endswith("Z"):
        s = s[:-1] + "+00:00"

    dt = datetime.fromisoformat(s)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def floor_to_minutes(dt: datetime, minutes: int) -> datetime:
    if minutes <= 0:
        raise ValueError("minutes must be positive")
    dt = dt.astimezone(timezone.utc).replace(second=0, microsecond=0)
    m = (dt.minute // minutes) * minutes
    return dt.replace(minute=m)


def ceil_to_minutes(dt: datetime, minutes: int) -> datetime:
    """
    Ceil dt to the next N-minute boundary (or keep if already aligned).
    """
    if minutes <= 0:
        raise ValueError("minutes must be positive")
    dt0 = dt.astimezone(timezone.utc).replace(second=0, microsecond=0)
    flo = floor_to_minutes(dt0, minutes)
    if flo == dt0:
        return dt0
    return flo + timedelta(minutes=minutes)


def iter_windows(start_dt: datetime, end_dt: datetime, minutes: int):
    """
    Yield (window_start, window_end) in UTC.
    Convention: [start, end) windows, last window end <= end_dt.
    """
    if end_dt <= start_dt:
        return
    cur = start_dt.astimezone(timezone.utc)
    end_dt = end_dt.astimezone(timezone.utc)
    step = timedelta(minutes=minutes)

    while cur < end_dt:
        nxt = cur + step
        yield cur, (nxt if nxt < end_dt else end_dt)
        cur = nxt


def to_iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Cell 4 — HTTP 请求（重试 + 限流）

In [0]:
# Reuse global session (connection pooling)
session = requests.Session()

# Optional: tune adapter pool size if needed
adapter = requests.adapters.HTTPAdapter(pool_connections=10, pool_maxsize=10)
session.mount("http://", adapter)
session.mount("https://", adapter)


def http_get_json(
    url: str,
    params: dict,
    timeout: tuple = (5, 30),  # (connect_timeout, read_timeout)
    max_retries: int = 5,
) -> Any:
    """
    HTTP GET with:
    - exponential backoff
    - jitter
    - 429 handling
    - Retry-After support
    - selective retry (5xx + network errors)
    """

    last_err = None

    for attempt in range(max_retries):
        try:
            response = session.get(url, params=params, timeout=timeout)

            # ---- Rate limit (429) ----
            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                if retry_after:
                    sleep_sec = min(60, int(retry_after))
                else:
                    sleep_sec = min(60, 2**attempt)

                sleep_sec += random.uniform(0, 1)  # jitter
                print(f"⚠️ 429 rate limit. Sleeping {sleep_sec:.2f}s")
                time.sleep(sleep_sec)
                continue

            # ---- Server errors (retryable) ----
            if 500 <= response.status_code < 600:
                sleep_sec = min(60, 2**attempt) + random.uniform(0, 1)
                print(f"⚠️ Server error {response.status_code}. Retry in {sleep_sec:.2f}s")
                time.sleep(sleep_sec)
                continue

            # ---- Client errors (non-retryable) ----
            if 400 <= response.status_code < 500:
                response.raise_for_status()

            response.raise_for_status()

            # ---- Parse JSON ----
            try:
                return response.json()
            except ValueError as je:
                raise RuntimeError(f"Invalid JSON response. url={url}, params={params}") from je

        except (requests.ConnectionError, requests.Timeout) as e:
            # network-level retry
            last_err = e
            sleep_sec = min(60, 2**attempt) + random.uniform(0, 1)
            print(f"⚠️ Network error. Retry in {sleep_sec:.2f}s")
            time.sleep(sleep_sec)

        except Exception as e:
            # unrecoverable error
            raise RuntimeError(f"HTTP fatal error. url={url}, params={params}, err={e}") from e

    raise RuntimeError(
        f"HTTP failed after {max_retries} retries. url={url}, params={params}, last_err={last_err}"
    )

# Cell 5 — Fetch：Coinbase 1m OHLC（按时间窗拉取）


In [0]:
COINBASE_CANDLES_URL = "https://api.exchange.coinbase.com/products/{product_id}/candles"
COINBASE_GRANULARITY_SEC = 60  # 1m


def _epoch_to_utc_dt(ts: int) -> datetime:
    return datetime.fromtimestamp(int(ts), tz=timezone.utc)


def _safe_float(x: Any, field: str, symbol: str) -> float:
    try:
        return float(x)
    except Exception as e:
        raise ValueError(f"Invalid {field} for {symbol}: {x}") from e


def fetch_ohlc_1m_coinbase(
    symbol: str,
    start_dt: datetime,
    end_dt: datetime,
    *,
    run_id: Optional[str] = None,
    window_start: Optional[datetime] = None,
    window_end: Optional[datetime] = None,
) -> List[Dict[str, Any]]:
    """
    Coinbase candles:
    GET /products/{product_id}/candles?start&end&granularity
    returns: [ [time, low, high, open, close, volume], ... ]
    time is epoch seconds (UTC)
    """

    # ---- normalize / validate time ----
    start_dt = start_dt.astimezone(timezone.utc)
    end_dt = end_dt.astimezone(timezone.utc)
    if end_dt <= start_dt:
        return []

    url = COINBASE_CANDLES_URL.format(product_id=symbol)
    params = {
        "start": to_iso_z(start_dt),
        "end": to_iso_z(end_dt),
        "granularity": COINBASE_GRANULARITY_SEC,
    }

    data = http_get_json(url, params=params)

    # ---- validate response shape ----
    if not isinstance(data, list):
        raise RuntimeError(
            f"Unexpected response type from Coinbase: {type(data)} | data={str(data)[:200]}"
        )

    ingest_ts = datetime.now(timezone.utc)

    rows: List[Dict[str, Any]] = []
    seen_ts = set()

    for x in data:
        # each x should be [time, low, high, open, close, volume]
        if not isinstance(x, (list, tuple)) or len(x) < 6:
            # skip bad row but do not silently ignore too much
            raise RuntimeError(f"Bad candle row from Coinbase: {x}")

        event_ts = int(x[0])
        if event_ts in seen_ts:
            continue
        seen_ts.add(event_ts)

        rows.append(
            {
                # --- dimensions ---
                "source": "coinbase",
                "symbol": symbol,
                "interval": "1m",
                # --- event time ---
                "event_ts": event_ts,  # epoch seconds (UTC)
                "event_time": _epoch_to_utc_dt(event_ts),  # timestamp (UTC)
                # --- measures ---
                "low": _safe_float(x[1], "low", symbol),
                "high": _safe_float(x[2], "high", symbol),
                "open": _safe_float(x[3], "open", symbol),
                "close": _safe_float(x[4], "close", symbol),
                "volume": _safe_float(x[5], "volume", symbol),
                # --- audit / lineage (recommended) ---
                "run_id": run_id,
                "ingest_ts": ingest_ts,
                "window_start": window_start or start_dt,
                "window_end": window_end or end_dt,
            }
        )

    # Coinbase may return in reverse order; normalize to ascending
    rows.sort(key=lambda r: r["event_ts"])
    return rows


def fetch_ohlc_1m(
    source: str,
    symbol: str,
    start_dt: datetime,
    end_dt: datetime,
    *,
    run_id: Optional[str] = None,
    window_start: Optional[datetime] = None,
    window_end: Optional[datetime] = None,
) -> List[Dict[str, Any]]:
    source = source.strip().lower()
    if source == "coinbase":
        return fetch_ohlc_1m_coinbase(
            symbol,
            start_dt,
            end_dt,
            run_id=run_id,
            window_start=window_start,
            window_end=window_end,
        )
    raise NotImplementedError(f"source={source} not implemented yet")

In [0]:
bucket_name = dbutils.widgets.get("s3_bucket").strip()
access_key = dbutils.widgets.get("aws_access_key").strip()
secret_key = dbutils.widgets.get("aws_secret_key").strip()
region = dbutils.widgets.get("aws_region").strip()
# region = "us-east-1"

if not bucket_name:
    raise ValueError("❌ s3_bucket is empty")
if not access_key or not secret_key:
    raise ValueError("❌ Missing AWS creds: aws_access_key / aws_secret_key widgets")
if not region:
    raise ValueError("❌ aws_region is empty (e.g. eu-central-1)")

_boto_sess = boto3.session.Session(
    aws_access_key_id=access_key, aws_secret_access_key=secret_key, region_name=region
)

s3 = _boto_sess.client("s3")

print("✅ S3 client ready. bucket:", bucket_name, "| region:", region)

# Cell 6 — S3 写入（JSONL，60 分钟一个文件

In [0]:
def _to_iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def _json_safe(obj: Any) -> Any:
    """
    Make objects JSON serializable (datetime -> ISO Z).
    Extend if you later introduce Decimal, numpy, etc.
    """
    if isinstance(obj, datetime):
        return _to_iso_z(obj)
    return obj


def build_s3_key(symbol: str, win_start: datetime, win_end: datetime) -> str:
    # Partition: dt/hour/win_start=HHMM (merge_minutes granularity)
    win_start = win_start.astimezone(timezone.utc)
    dt_str = win_start.strftime("%Y-%m-%d")
    hour_str = win_start.strftime("%H")
    hhmm = win_start.strftime("%H%M")

    # Example:
    # crypto/ohlc_1m/source=coinbase/symbol=BTC-USD/dt=2026-02-11/hour=13/win_start=1315/part.jsonl.gz
    return (
        f"{s3_prefix}/"
        f"source={source}/symbol={symbol}/dt={dt_str}/hour={hour_str}/"
        f"win_start={hhmm}/"
        f"ohlc_1m_{dt_str}T{hhmm}00Z_{merge_minutes}m.jsonl.gz"
    )


def upload_jsonl_gz(bucket: str, key: str, rows: List[Dict[str, Any]]) -> int:
    # Skip empty windows (optional but recommended to avoid useless small files)
    if not rows:
        return 0

    buf = io.BytesIO()
    with gzip.GzipFile(fileobj=buf, mode="wb") as gz:
        for r in rows:
            line = json.dumps(r, ensure_ascii=False, default=_json_safe) + "\n"
            gz.write(line.encode("utf-8"))

    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=buf.getvalue(),
        ContentType="application/x-ndjson",
        ContentEncoding="gzip",
    )
    return len(rows)


def produce_one_window(symbol: str, win_start: datetime, win_end: datetime) -> Dict[str, Any]:
    # File-level metadata (stable within one file)
    run_id = str(uuid.uuid4())
    produced_at = datetime.now(timezone.utc)

    rows = fetch_ohlc_1m(
        source,
        symbol,
        win_start,
        win_end,
        run_id=run_id,
        window_start=win_start,
        window_end=win_end,
    )

    # Enrich (keep naming consistent; avoid duplicating fields from Cell 5)
    enriched = []
    for r in rows:
        enriched.append(
            {
                **r,
                "_producer_run_id": run_id,
                "_produced_at": produced_at,  # datetime OK, _json_safe handles it
                "_merge_minutes": merge_minutes,
            }
        )

    # Optional: window quality check (15m should ~15 rows)
    expected = merge_minutes  # for 1m bars
    if enriched and len(enriched) < max(1, expected // 2):
        print(
            f"⚠️ Low row count for window {symbol} {win_start}->{win_end}: {len(enriched)} (expected ~{expected})"
        )

    key = build_s3_key(symbol, win_start, win_end)
    n = upload_jsonl_gz(s3_bucket, key, enriched)

    return {
        "symbol": symbol,
        "window_start": _to_iso_z(win_start),
        "window_end": _to_iso_z(win_end),
        "s3_key": key,
        "rows": n,
        "run_id": run_id,
    }

# Cell 7 — Backfill 模式（按 15 分钟窗口回补）

In [0]:
def validate_backfill_range(start_dt: datetime, end_dt: datetime, max_days: int):
    if end_dt <= start_dt:
        raise ValueError(f"Backfill range invalid: end_dt <= start_dt ({start_dt} -> {end_dt})")
    delta_days = (end_dt - start_dt).total_seconds() / 86400.0
    if delta_days > max_days:
        raise ValueError(f"Backfill range too large: {delta_days:.2f} days > max_days={max_days}")


def run_backfill():
    if not start_ts_raw or not end_ts_raw:
        raise ValueError("❌ backfill mode requires start_ts and end_ts widgets")

    start_dt = parse_utc_iso(start_ts_raw)
    end_dt = parse_utc_iso(end_ts_raw)

    validate_backfill_range(start_dt, end_dt, max_days)

    # Align to merge boundary
    start_aligned = floor_to_minutes(start_dt, merge_minutes)
    end_aligned = ceil_to_minutes(end_dt, merge_minutes)

    print(f"🔵 Backfill aligned range (UTC): {to_iso_z(start_aligned)} -> {to_iso_z(end_aligned)}")
    print(f"   symbols={symbols} | merge_minutes={merge_minutes} | sleep_ms={sleep_ms}")

    # Pre-calc total windows for progress display
    total_windows = 0
    for _w0, _w1 in iter_windows(start_aligned, end_aligned, merge_minutes):
        total_windows += 1
    total_tasks = total_windows * len(symbols)

    # Stats
    ok = 0
    empty = 0
    failed = 0
    failures: List[Dict[str, Any]] = []
    last_n: List[Dict[str, Any]] = []  # keep only last N results
    KEEP_LAST = 5

    t0 = time.time()
    done = 0

    for sym in symbols:
        for w0, w1 in iter_windows(start_aligned, end_aligned, merge_minutes):
            done += 1
            try:
                res = produce_one_window(sym, w0, w1)
                # res["rows"] == 0 means skipped/empty window
                if res.get("rows", 0) == 0:
                    empty += 1
                else:
                    ok += 1

                # keep last few
                last_n.append(res)
                if len(last_n) > KEEP_LAST:
                    last_n = last_n[-KEEP_LAST:]

            except Exception as e:
                failed += 1
                failures.append(
                    {
                        "symbol": sym,
                        "window_start": to_iso_z(w0),
                        "window_end": to_iso_z(w1),
                        "error": str(e)[:500],
                    }
                )

            # throttle between windows (smooth API usage)
            if sleep_ms > 0:
                time.sleep(sleep_ms / 1000.0)

            # progress every 20 windows (or at end)
            if done % 20 == 0 or done == total_tasks:
                elapsed = time.time() - t0
                rate = done / elapsed if elapsed > 0 else 0.0
                remaining = total_tasks - done
                eta_sec = remaining / rate if rate > 0 else float("inf")
                print(
                    f"📈 Progress: {done}/{total_tasks} | ok={ok} empty={empty} fail={failed} | "
                    f"rate={rate:.2f} win/s | ETA={eta_sec / 60:.1f} min"
                )

    print("✅ Backfill done.")
    print(f"   Total tasks={total_tasks} | ok={ok} empty={empty} fail={failed}")

    print("Last results:")
    for r in last_n:
        print(r)

    if failures:
        print("⚠️ Failures (first 20):")
        for f in failures[:20]:
            print(f)

    return {"ok": ok, "empty": empty, "failed": failed, "failures": failures}


# Execute only in backfill mode
if mode == "backfill":
    backfill_summary = run_backfill()

# Cell 8 — Realtime 模式（持续产出“已完成的”15 分钟窗口文件）

In [0]:
def compute_realtime_target_window(
    now_utc: datetime, merge_minutes: int, safety_lag_minutes: int
) -> Tuple[datetime, datetime]:
    """
    Emit the most recent completed window:
    safe_now = now - safety_lag
    floor_dt = floor(safe_now, merge_minutes)
    target window: [floor_dt - merge, floor_dt)
    """
    safe_now = now_utc - timedelta(minutes=safety_lag_minutes)
    floor_dt = floor_to_minutes(safe_now, merge_minutes)
    win_end = floor_dt
    win_start = win_end - timedelta(minutes=merge_minutes)
    return win_start, win_end


def s3_key_exists(bucket: str, key: str) -> bool:
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except Exception:
        return False


def get_missing_windows(
    latest_start: datetime, latest_end: datetime, max_catchup: int
) -> List[Tuple[datetime, datetime]]:
    """
    If we are behind, generate up to max_catchup windows ending at latest_end.
    Returns windows in chronological order.
    """
    windows = []
    # backfill-like: (latest_start, latest_end) is "current target"
    # We generate up to max_catchup windows backwards then reverse.
    w0, w1 = latest_start, latest_end
    for _ in range(max_catchup):
        windows.append((w0, w1))
        w1 = w0
        w0 = w1 - timedelta(minutes=merge_minutes)
    windows.reverse()
    return windows


def run_realtime():
    loops = 0
    last_target_end = None  # for quiet logging

    # catch-up limit: prevent infinite backlog
    MAX_CATCHUP_WINDOWS = 8  # e.g., up to 2 hours if merge=15m

    while True:
        now_utc = datetime.now(timezone.utc)
        latest_w0, latest_w1 = compute_realtime_target_window(
            now_utc, merge_minutes, safety_lag_minutes
        )

        # quiet log: only when target window changes
        if last_target_end != latest_w1:
            print(
                f"🟢 Latest completed window target: {to_iso_z(latest_w0)} -> {to_iso_z(latest_w1)}"
            )
            last_target_end = latest_w1

        # catch-up strategy: attempt a small backlog of recent windows (chronological)
        windows_to_try = get_missing_windows(latest_w0, latest_w1, MAX_CATCHUP_WINDOWS)

        for w0, w1 in windows_to_try:
            # per window, attempt each symbol with strong idempotency
            all_ok = True

            for sym in symbols:
                key = build_s3_key(sym, w0, w1)

                # if already exists, skip (strong idempotency across restarts)
                if s3_key_exists(s3_bucket, key):
                    # optional: print occasionally; keep quiet by default
                    continue

                try:
                    print(f"📦 Producing: {sym} | {to_iso_z(w0)} -> {to_iso_z(w1)}")
                    res = produce_one_window(sym, w0, w1)
                    print("   ✅ WROTE:", res)
                except Exception as e:
                    all_ok = False
                    print(f"   ❌ ERROR: {sym} {to_iso_z(w0)}->{to_iso_z(w1)} | {e}")

                # smooth throttle between symbol calls
                if sleep_ms > 0:
                    time.sleep(sleep_ms / 1000.0)

            # If not all_ok, we do NOT "mark emitted" anywhere:
            # next loop will re-attempt missing keys (since head_object will still be false).
            if not all_ok:
                # avoid hammering; short pause before moving on
                time.sleep(2)

        loops += 1
        if max_loops > 0 and loops >= max_loops:
            print("🛑 Reached max_loops. Stop.")
            break

        time.sleep(loop_sleep_seconds)


if mode == "realtime":
    run_realtime()

In [0]:
%skip
datetime.now(timezone.utc).isoformat()